In [1]:
#import

import random
import cv2
import json
import numpy as np
import copy
import pandas as pd
import torch
import gc

import albumentations as A
import os
import librosa
import pickle
import timm
from tqdm import tqdm
import mne

import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import DataLoader

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
CFG={
    'batch_size':32,
    'num_worker':4,
    'data':'/kaggle/input/hms-harmful-brain-activity-classification/test.csv',
    'weights_spec':'/kaggle/input/hms-baseline',
    'weights_eeg':'/kaggle/input/hms-eeg',
    'weights_mix':'/kaggle/input/hms-mix',
    
    'flip':True
}

In [3]:
CFG['weights_spec']=[os.path.join(CFG['weights_spec'],x) for x in sorted(os.listdir(CFG['weights_spec']))]
CFG['weights_eeg']=[os.path.join(CFG['weights_eeg'],x) for x in sorted(os.listdir(CFG['weights_eeg']))]
CFG['weights_mix']=[os.path.join(CFG['weights_mix'],x) for x in sorted(os.listdir(CFG['weights_mix']))]

CFG

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/hms-baseline'

In [4]:
#make eeg specs
import os
import pickle

import librosa
import numpy as np
import pandas as pd
import torch
import torchaudio
from torch import nn
from tqdm import tqdm
data_dir='/kaggle/input/hms-harmful-brain-activity-classification/test_eegs'
NAMES = ['LL', 'LP', 'RP', 'RR']

FEATS = [['Fp1', 'F7', 'T3', 'T5', 'O1'],
         ['Fp1', 'F3', 'C3', 'P3', 'O1'],
         ['Fp2', 'F8', 'T4', 'T6', 'O2'],
         ['Fp2', 'F4', 'C4', 'P4', 'O2']]



class Transform(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.wave_transform = torchaudio.transforms.MelSpectrogram(sample_rate=200,
                                                          hop_length=10000// 256,
                                                          n_fft=1024,
                                                          n_mels=128,
                                                          f_min=0, f_max=20,
                                                          win_length=128)

        # self.am2db= torchaudio.transforms.AmplitudeToDB(stype='power',
        #                                                 top_db=80)


    def forward(self, x):
        bs = x.size(0)


        image = self.wave_transform(x)
        # image = self.am2db(image)
        return image


transform_func=Transform().to('cuda')
def spectrogram_from_eeg( parquet_path, display=False):
    # LOAD MIDDLE 50 SECONDS OF EEG SERIES
    eeg = pd.read_parquet(parquet_path)
    middle = (len(eeg) - 10_000) // 2
    eeg = eeg.iloc[middle:middle + 10_000]

    # VARIABLE TO HOLD SPECTROGRAM
    img = np.zeros((128, 256, 4), dtype='float32')

    signals = []
    for k in range(4):
        COLS = FEATS[k]

        for kk in range(4):

            # COMPUTE PAIR DIFFERENCES
            x = eeg[COLS[kk]].values - eeg[COLS[kk + 1]].values

            # FILL NANS
            m = np.nanmean(x)
            if np.isnan(x).mean() < 1:
                x = np.nan_to_num(x, nan=m)
            else:
                x[:] = 0


            x_tensor=torch.from_numpy(x).to('cuda')
            mel_spec=transform_func(x_tensor)
            mel_spec=mel_spec.cpu().numpy()
            # LOG TRANSFORM

            width = (mel_spec.shape[1] // 32) * 32
            mel_spec_db = librosa.power_to_db(mel_spec,
                                              ref=np.max).astype(np.float32)[:, :width]

            
            img[:, :, k] += mel_spec_db
            
        # AVERAGE THE 4 MONTAGE DIFFERENCES
        img[:, :, k] /= 4.0

    return img


all_fs=os.listdir(data_dir)
all_specs={}



for item in tqdm(all_fs):

    eeg_id=item.rsplit('.',1)[0]

    eeg_path = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/%s.parquet' % (eeg_id)



    eeg_spec=spectrogram_from_eeg(eeg_path)

    all_specs[eeg_id]=eeg_spec




with open('eeg_specs_dict.pkl', 'wb') as file:
    pickle.dump(all_specs, file)




RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [5]:
#dataiter
class AlaskaDataIter():
    def __init__(self, df,
                 training_flag=False,shuffle=False,
                 use_spec=False,
                 use_eeg=False,
                 use_mix=False,
                 ll=0,rr=20):
        
        
        self.ll=0
        self.rr=rr
        self.training_flag = training_flag
        self.shuffle = shuffle

        self.raw_data_set_size = None     ##decided by self.parse_file


        self.df=df
        
        self.train_trans=A.Compose([

                                    A.HorizontalFlip(p=0.5),

                              ])

        TARS = {'Seizure': 0, 'LPD': 1, 'GPD': 2, 'LRDA': 3, 'GRDA': 4, 'Other': 5}
        self.TARS2 = {x: y for y, x in TARS.items()}


        self.eeg_nms=['Fp1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'Fz', 'Cz', 'Pz',
       'Fp2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2', 'EKG']

        self.LL = ['Fp1', 'F7', 'T3', 'T5', 'O1']

        self.RR = ['Fp2', 'F8', 'T4', 'T6', 'O2']

        self.LP = ['Fp1', 'F3', 'C3', 'P3', 'O1']

        self.RP = ['Fp2', 'F4', 'C4', 'P4', 'O2']

        self.leads_dict = {value: index for index, value in enumerate(self.eeg_nms)}

        self.use_eeg = use_eeg
        self.use_spec = use_spec
        self.use_mix = use_mix
        
        if self.use_mix:
            with open('eeg_specs_dict.pkl', mode='rb') as f:
                self.eeg_specs = pickle.load(f)
        
        
    def __getitem__(self, item):

        return self.single_map_func(self.df.iloc[item], self.training_flag)

    def __len__(self):

        return len(self.df)

    def brain_lead(self, waves):
        waves = copy.deepcopy(waves)

        brain_leads = [self.LL,self.LP,self.RP,self.RP]


        leads = []

        for combine in brain_leads:
            for i in range(len(combine)-1):

                tmp_lead = waves[self.leads_dict[combine[i]]] - waves[self.leads_dict[combine[i + 1]]]
                leads.append(tmp_lead)



        data = np.concatenate([leads], axis=0)


        return data
    
    def get_eeg(self, dp, is_training):
        eeg_path = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/%s.parquet' % (dp['eeg_id'])
        eeg = pd.read_parquet(eeg_path)

        
        offset = 0
        eeg = eeg.iloc[int(offset * 200):int(offset * 200) + 10000]

        waves = eeg.values

        waves = np.transpose(waves, axes=[1, 0])

        for i in range(waves.shape[0]):
            m = np.nanmean(waves[i])
            if np.isnan(waves[i]).mean() < 1:
                waves[i] = np.nan_to_num(waves[i], nan=m)
            else:
                waves[i] = 0

        waves = np.array(waves, dtype=np.float64)
        waves = mne.filter.filter_data(waves, 200, self.ll, self.rr, verbose=False)

        waves = self.brain_lead(waves)

        return waves
    def get_spec(self, dp, is_training):
        spec_path = '/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/%s.parquet' % (dp['spectrogram_id'])
        spec = pd.read_parquet(spec_path)

        spec=spec.values[:,1:]

        images=[]
        r = 0

        for region in range(4):
            img = spec[r:r + 300,
                  region * 100:(region + 1) * 100].T
            img = np.clip(img, np.exp(-4), np.exp(8))
            img = np.log(img)

            # STANDARDIZE PER IMAGE
            # ep = 1e-6
            # m = np.nanmean(img.flatten())
            # s = np.nanstd(img.flatten())
            # img = (img - m) / (s + ep)
            img = np.nan_to_num(img, nan=0.0)

            images.append(img)

        images=np.stack(images,-1)
        data = np.transpose(images, [2,0,1])


        return data
    
    def get_mix(self, dp, is_training):
        X = np.zeros((128, 256, 8), dtype='float32')
        kg_spec = self.get_spec(dp, is_training=False)

        eeg_spec = self.eeg_specs[str(dp['eeg_id'])]
        
        kg_spec=np.transpose(kg_spec,axes=[1,2,0])
        
        X[ 14:-14, :, :4] = kg_spec[:, 22:-22]
        X[:, :, 4:] = eeg_spec

        X = np.transpose(X, [2, 0, 1])
        return X
    def single_map_func(self, dp, is_training):
        """Data augmentation function."""
        ####customed here

        
        if self.use_eeg:
            data=self.get_eeg(dp,is_training)
        elif self.use_spec:
            data=self.get_spec(dp,is_training)
        elif self.use_mix:
            data=self.get_mix(dp,is_training)

        
        return data.astype(np.float32)

In [6]:



class NetSpec(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()

        

        self.model = timm.create_model('efficientnet_b5',
                                       pretrained=False,
                                       in_chans=3)

        self.fc = nn.Linear(2048, 6, bias=True)
        # self.wave_encoder = ResNet1D(inp_ch=42, block=ResNetBlock1D, layers=[2, 2, 4, 2], num_classes=1)

        # weight_init(self.fc)
        self.dropout=nn.Dropout(0.5)

        self.avg=nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        # do preprocess
        bs = x.size(0)

        # wave_fm = self.wave_encoder(x)  # 1x250
        # wave_fm = wave_fm.unsqueeze(2)
        #
        x1 = [x[:,  i:i + 1,:, :] for i in range(4)]
        x1 = torch.cat(x1,dim=2)
        x= torch.cat([x1,x1,x1],dim=1)
        

        if CFG['flip']:
            x_flip=torch.flip(x, [3])
            x=torch.cat([x,x_flip],dim=0)

            

        x = self.model.forward_features(x)
        x = self.avg(x)

        if CFG['flip']:
            x = x.view(2*bs, -1)
        else:
            x = x.view(bs, -1)
        x = self.dropout(x)

        x = self.fc(x)
        
        x =torch.softmax(x,-1)
        
        if CFG['flip']:
            ans= (x[:bs,...]+x[bs:,...])/2.
        else:
            ans=x
        return ans




In [7]:

class Transform(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.wave_transform = torchaudio.transforms.Spectrogram(n_fft=512, hop_length=25, power=1)

        self.am2db= torchaudio.transforms.AmplitudeToDB(stype='magnitude',top_db=80)

        self.resizer=nn.UpsamplingBilinear2d(size=[160,320])

    def forward(self, x):
        bs = x.size(0)


        image = self.wave_transform(x)
        image = self.am2db(image)
        n, c, h, w = image.size()
        image=image[:,:,:int(20/100*h+30),:]

        image = torch.reshape(image,shape=[n,4,-1,w])




        return image




class NetEeg(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()

        self.preprocess = Transform()

        self.model = timm.create_model('efficientnet_b5',
                                       pretrained=False,
                                       in_chans=4)

        self.fc = nn.Linear(2048, 6, bias=True)
        # self.wave_encoder = ResNet1D(inp_ch=42, block=ResNetBlock1D, layers=[2, 2, 4, 2], num_classes=1)

        # weight_init(self.fc)
        self.dropout=nn.Dropout(0.5)

        self.avg=nn.AdaptiveAvgPool2d(1)


    def forward(self, x):
        # do preprocess
        bs = x.size(0)

        x= self.preprocess(x)
        # wave_fm = self.wave_encoder(x)  # 1x250
        # wave_fm = wave_fm.unsqueeze(2)
        #
        # x1 = [x[:, i:i + 1, :, :] for i in range(4)]
        # x1 = torch.cat(x1, dim=2)
        # x = torch.cat([x1, x1, x1], dim=1)
        
        if CFG['flip']:
            x_flip=torch.flip(x, [3])
            x=torch.cat([x,x_flip],dim=0)
            
            
        x = self.model.forward_features(x)
        x = self.avg(x)
        if CFG['flip']:
            x = x.view(2*bs, -1)
        else:
            x = x.view(bs, -1)
        x = self.dropout(x)

        x = self.fc(x)
        x =torch.softmax(x,-1)
        
        if CFG['flip']:
            ans= (x[:bs,...]+x[bs:,...])/2.
        else:
            ans=x
            
        return ans


In [8]:

class NetMix(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()

        self.preprocess = Transform()

        self.model = timm.create_model('hrnet_w18',
                                       pretrained=False,
                                       in_chans=3)

        self.fc = nn.Linear(2048, 6, bias=True)
        # self.wave_encoder = ResNet1D(inp_ch=42, block=ResNetBlock1D, layers=[2, 2, 4, 2], num_classes=1)

        # weight_init(self.fc)
        self.dropout=nn.Dropout(0.5)

        self.avg=nn.AdaptiveAvgPool2d(1)


    def forward(self, x):
        # do preprocess
        bs = x.size(0)
        if CFG['flip']:
            x_flip=torch.flip(x, [3])
            x=torch.cat([x,x_flip],dim=0)
            
        x1 = [x[:, i:i + 1, :, :] for i in range(4)]
        x1 = torch.cat(x1, dim=2)

        x2=[x[:, i+4:i + 5, :, :] for i in range(4)]
        x2 = torch.cat(x2, dim=2)

        x= torch.cat([x1,x2],dim=3)

        x = torch.cat([x, x, x], dim=1)
        

            
        x = self.model.forward_features(x)
        x = self.avg(x)
        
        if CFG['flip']:
            x = x.view(2*bs, -1)
        else:
            x = x.view(bs, -1)
        
        x = self.dropout(x)
        x = self.fc(x)

        x =torch.softmax(x,-1)

        if CFG['flip']:
            ans= (x[:bs,...]+x[bs:,...])/2.
        else:
            ans=x
            
        return ans



In [9]:
def inference_function(test_loader, model, device):
    model.eval()
    
    prediction_dict = {}
    preds = []
    with tqdm(test_loader, unit="test_batch", desc='Inference') as tqdm_test_loader:
        for step, X in enumerate(tqdm_test_loader):
            X = X.to(device)
            
            
            batch_size = X.size(0)
            with torch.no_grad():
                y_preds = model(X)

            preds.append(y_preds.to('cpu').numpy()) 
                
    prediction_dict["predictions"] = np.concatenate(preds) 
    return prediction_dict

In [10]:
test_df=pd.read_csv(CFG['data'])

test_df.head(5)


,spectrogram_id,eeg_id,patient_id
0,2207717,2578018731,34153
1,2207717,2578018731,34153
2,2207717,2578018731,34153
3,2207717,2578018731,34153
4,8352559,1706196508,37552


In [11]:
predictions = []
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# print('infer with weights_spec')
# for model_weight in CFG['weights_spec']:
#     test_dataset = AlaskaDataIter(test_df, training_flag=False, shuffle=False,use_spec=True)
#     test_loader = DataLoader(test_dataset,
#                      CFG['batch_size'],
#                      num_workers=CFG['num_worker'],
#                      shuffle=False)

#     model = NetSpec()
#     state_dict = torch.load(model_weight, map_location=device)

#     model.load_state_dict(state_dict,strict=False)
#     model.to(device)
#     prediction_dict = inference_function(test_loader, model, device)
#     predictions.append(prediction_dict["predictions"])
#     torch.cuda.empty_cache()
#     gc.collect()
    
print('infer with weights_eeg')
for model_weight in CFG['weights_eeg']:
    test_dataset = AlaskaDataIter(test_df, training_flag=False, shuffle=False,use_eeg=True)
    test_loader = DataLoader(test_dataset,
                     CFG['batch_size'],
                     num_workers=CFG['num_worker'],
                     shuffle=False)

    model = NetEeg()
    state_dict = torch.load(model_weight, map_location=device)

    model.load_state_dict(state_dict,strict=False)
    model.to(device)
    prediction_dict = inference_function(test_loader, model, device)
    predictions.append(prediction_dict["predictions"])
    torch.cuda.empty_cache()
    gc.collect()
    
# print('infer with weights_mix')
# for model_weight in CFG['weights_mix']:
#     test_dataset = AlaskaDataIter(test_df, training_flag=False, shuffle=False,use_mix=True)
#     test_loader = DataLoader(test_dataset,
#                      CFG['batch_size'],
#                      num_workers=CFG['num_worker'],
#                      shuffle=False)

#     model = NetMix()
#     state_dict = torch.load(model_weight, map_location=device)

#     model.load_state_dict(state_dict,strict=False)
#     model.to(device)
#     prediction_dict = inference_function(test_loader, model, device)
#     predictions.append(prediction_dict["predictions"])
#     torch.cuda.empty_cache()
#     gc.collect()



    
predictions = np.array(predictions)
predictions = np.mean(predictions, axis=0)

predictions.shape

infer with weights_eeg


IsADirectoryError: [Errno 21] Is a directory: '/'

In [12]:
TARGETS = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
sub = pd.DataFrame({'eeg_id': test_df.eeg_id.values})
sub[TARGETS] = predictions
sub.to_csv('submission.csv',index=False)
print(f'Submissionn shape: {sub.shape}')
sub.head()

ValueError: Columns must be same length as key